# Comparison of different regression models

On the diabetes dataset (scikit-learn), different lienar regression models are compared with non-linear models. Discuss the impact of the different loss functions for ``Lasso``, ``Ridge`` and ``ElasticNet``.

Discuss model efficiency and interpretation of various metrics.

On the data: The diabetes dataset is a classic "toy" dataset for regression benchmarks. It includes physiological features such as the age, bmi or some blood serum values (s1 - s6), all features are z-scaled. The target is a metric for the progression of the disease.


In [ ]:
from sklearn.datasets import load_diabetes
import pandas as pd

data = load_diabetes()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print(X.shape)
print(X.head())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

# alpha (regularisation parameter in the loss function to penalise coefficients)
# hyperparameters on linear model will be optimised later
lin_models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5)
}

# hyperparameters for non-linear methods as suggested by Copilot
non_lin_models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        bootstrap=True,
        random_state=42
        ),
    "HistGB": HistGradientBoostingRegressor(     
        learning_rate=0.05,
        max_depth=5,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=1.0,
        max_bins=255,
        early_stopping=True,
        random_state=42
        ),
    "SVR": SVR(    
        kernel="rbf",
        C=10,
        gamma="scale",
        epsilon=0.1
        )
}

models = lin_models | non_lin_models # concatenation of two dicionaries

In [ ]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import time

results_list = []

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - t0
    y_pred = model.predict(X_test)

    results = {
        "model": name,
        "test_RMSE": root_mean_squared_error(y_test, y_pred),
        "test_MAE": mean_absolute_error(y_test, y_pred),
        "test_R2": r2_score(y_test, y_pred),
        "fit_time_s": fit_time
    }
    results_list.append(results)

res_df = pd.DataFrame(results_list).sort_values("test_RMSE")
res_df

In [ ]:
# investigate the linear model weights / coefficients

coef_df = pd.DataFrame({
    name: model.fit(X_train, y_train).coef_
    for name, model in lin_models.items()
}, index=X.columns)

print(coef_df)

In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV

ridge = RidgeCV(alphas=[0.1, 1, 10])
lasso = LassoCV(cv=5)
elastic = ElasticNetCV(cv=5)

ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)
elastic.fit(X_train, y_train)

print("Best Ridge alpha:", ridge.alpha_)
print("Best Lasso alpha:", lasso.alpha_)
print("Best ElasticNet alpha:", elastic.alpha_)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

y_pred = lin_models["Lasso"].predict(X_test)

In [ ]:
sns.scatterplot(x=y_test, y=y_pred)

# Identity line

lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, "r--", lw=1)

plt.xlabel("True")
plt.ylabel("Predicted")
plt.title("Model predictions with identity line")
plt.grid(alpha=0.3)
plt.show()
